### Import Dependencies

In [ ]:
import openai
import pandas as pd
import tiktoken

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import FieldCondition, MatchAny, Filter
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery 

### Create Qdrant collection for hybrid search

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-reviews-collection-01",
    vectors_config={
        "text-embedding-3-small":VectorParams(size=1536, distance=Distance.COSINE)
    }
)

In [ ]:
qdrant_client.create_payload_index(
    collection_name="Amazon-reviews-collection-01",
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
)

### Embedding Function

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [ ]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

### Read the sampled dataset with Amazon inventory data

In [ ]:
df_reviews = pd.read_json("../../data/Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
df_reviews.head()

In [ ]:
len(df_reviews)

### Preprocess title and features

In [ ]:
def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

In [ ]:
def count_tokens(row):

    encoding = tiktoken.encoding_for_model("text-embedding-3-small")

    return len(encoding.encode(row["preprocessed_data"]))

In [ ]:
df_reviews["preprocessed_data"] = df_reviews.apply(preprocess_reviews_data, axis=1)
df_reviews["token_count"] = df_reviews.apply(count_tokens, axis=1)

In [ ]:
df_reviews.head()

In [ ]:
len(df_reviews)

In [ ]:
df_reviews = df_reviews[df_reviews["token_count"] < 8192]

In [ ]:
len(df_reviews)

In [ ]:
total_tokens = df_reviews["token_count"].sum()

In [ ]:
total_tokens

### Embed the text and add additional fields to the payload of each vector for reviews

In [ ]:
df_data_to_embed = df_reviews[["preprocessed_data", "parent_asin"]]

In [ ]:
df_data_to_embed.head()

In [ ]:
data_to_embed_reviews = df_data_to_embed.to_dict(orient="records")

In [ ]:
data_to_embed_reviews

In [ ]:
len(data_to_embed_reviews)

In [ ]:
text_to_embed_reviews = [item["preprocessed_data"] for item in data_to_embed_reviews]

In [ ]:
text_to_embed_reviews

In [ ]:
embeddings = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

In [ ]:
len(embeddings)

In [ ]:
pointstructs = []
i=1
for embedding, data in zip(embeddings, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
            },
            payload=data
        )
    )
    i += 1

In [ ]:
pointstructs[0].vector

In [ ]:
batch_size_qdrant = 100
counter = 1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i + batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-reviews-collection-01",
        points=batch,
        wait=True
    )
    print(f"Processed {counter * batch_size_qdrant} of {len(pointstructs)}")
    counter += 1

### A function to run search against reviews on a pre-filtered set of product ID

In [ ]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)


    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
         query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [ ]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B09Q5TNDHY"])

In [ ]:
reviews

In [ ]:
reviews.points

In [ ]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B09Q5TNDHY", "B0B4NJ8NKN", ])

In [ ]:
reviews

In [ ]:
reviews.points

### Define the reviews retrieval tool

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    qdrant_client = QdrantClient(url="http://localhost:6333")

    results = qdrant_client.query_points(
        collection_name="Amazon-reviews-collection-01",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
         query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []


    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_contexts.append(result.payload["preprocessed_data"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
    }


def process_context_reviews(context):
    
    formated_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_contexts"]):
        formated_context += f"- ID: {id}, user review: {chunk}\n"

    return formated_context


def get_formatted_reviews_context(query: str, parent_asins: list[str], top_k: int = 5) -> str:

    """Get the top k reviews matching a query for a list of prefiltered items.

    Args:
        query: The query to get the top k reviews for
        item_list: The list of item IDs to prefilter for before running the query
        top_k: The number of reviews to retrieve, this should be at least 20 if multipple items are prefiltered

    Returns:
        A string of the top k context chunks with IDs prepending each chunk, each representing a review for a given inventory item for a given query.
    """

    retrieved_context = retrieve_prefiltered_reviews_data(
        query,
        parent_asins,
        k=20
    )

    formatted_context = process_context_reviews(retrieved_context)

    return formatted_context

In [ ]:
result = get_formatted_reviews_context("bad quality", ["B09Q5TNDHY", "B0B4NJ8NKN"])

In [ ]:
print(result)